# XGBpure — Elos as Direct Features (No Base Margin)

Compare two approaches using the **locked hyperparams** from `XGB_tuning.ipynb`:

| Approach | How Elo enters the model |
|---|---|
| **Margin-base** (tuning notebook) | `base_margin = logit(p_elo)` passed to XGBoost as prior |
| **Elo-as-features** (this notebook) | `home_elo_pre` and `away_elo_pre` added to feature matrix; no `base_margin` |

Walk-forward CV: train 2015...(Y-1), test Y, for Y in {2020..2024}.  
Reports per-fold and mean log-loss, accuracy, Brier score.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path
from sklearn.metrics import log_loss, brier_score_loss

_cwd = Path.cwd()
if (_cwd / "data").exists() and (_cwd / "notebooks").exists():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "data").exists() and (_cwd.parent / "notebooks").exists():
    PROJECT_ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot find project root from cwd={_cwd}")

GOLD_DIR = PROJECT_ROOT / "data" / "gold" / "stage1" / "hM7_Linj14_tau150_hT7"

# Locked hyperparams (XGB_tuning.ipynb Stage 3 winner)
XGB_PARAMS = {
    "objective":        "binary:logistic",
    "eval_metric":      "logloss",
    "max_depth":        6,
    "min_child_weight": 3,
    "subsample":        0.9,
    "colsample_bytree": 0.6,
    "reg_lambda":       1,
    "reg_alpha":        0.0,
    "gamma":            1,
    "learning_rate":    0.02,
    "seed":             42,
    "nthread":          -1,
}
N_PLAYERS       = 7
EARLY_STOPPING  = 150
NUM_BOOST_ROUND = 3000

PLAYER_FEATS   = ["m_ewma_pre", "q_pre", "days_since_first_report_pre",
                  "days_since_last_dnp_pre", "consec_dnps_pre",
                  "played_last_game_pre", "minutes_last_game_pre",
                  "days_since_last_played_pre", "injury_present_flag_pre"]
FORM_FEATS     = ["net_rtg_ewma_pre", "efg_ewma_pre", "tov_pct_ewma_pre",
                  "orb_pct_ewma_pre", "ftr_ewma_pre"]
STYLE_FEATS    = ["off_3pa_rate_pre", "def_3pa_allowed_pre",
                  "off_2pa_rate_pre", "def_2pa_allowed_pre",
                  "off_tov_pct_pre", "def_forced_tov_pre"]
SCHEDULE_FEATS = ["days_rest_pre", "is_b2b_pre", "games_last_4_days_pre",
                  "games_last_7_days_pre", "travel_miles_pre",
                  "timezone_shift_hours_pre"]

def build_feature_cols(n):
    cols = [f"{side}_p{slot}_{f}"
            for side in ("home", "away")
            for slot in range(1, n + 1)
            for f in PLAYER_FEATS]
    for f in FORM_FEATS + STYLE_FEATS + SCHEDULE_FEATS:
        cols += [f"home_{f}", f"away_{f}"]
    return cols

BASE_FEATURE_COLS = build_feature_cols(N_PLAYERS)
# Add elos directly as features — no base_margin
FEATURE_COLS = BASE_FEATURE_COLS + ["home_elo_pre", "away_elo_pre"]

def load_year(year):
    p = GOLD_DIR / f"game_xgboost_input_{year}_REGPST.csv"
    df = pd.read_csv(p)
    cold = (df["home_p1_m_ewma_pre"] == 0) | (df["away_p1_m_ewma_pre"] == 0)
    df = df[~cold].dropna(subset=["home_win"]).reset_index(drop=True)
    return df

def to_dmatrix(df, feats):
    avail = [c for c in feats if c in df.columns]
    X = df[avail].values.astype(float)
    y = df["home_win"].values.astype(float)
    return xgb.DMatrix(X, label=y, feature_names=avail, missing=np.nan)

print(f"Feature count: {len(FEATURE_COLS)} ({len(BASE_FEATURE_COLS)} base + home_elo_pre + away_elo_pre)")
print("base_margin: NOT used")


Feature count: 162 (160 base + home_elo_pre + away_elo_pre)
base_margin: NOT used


In [2]:
# Walk-forward CV
# For each test year Y in {2020..2024}:
#   1. Train on 2015..Y-2, early-stop val on Y-1 -> best_round
#   2. Retrain on 2015..Y-1 for best_round rounds
#   3. Evaluate on Y

TEST_YEARS  = [2020, 2021, 2022, 2023, 2024]
TRAIN_START = 2015
results = []

print(f"{'Year':>6} {'n_test':>7} {'LogLoss':>10} {'Accuracy':>10} {'Brier':>10} {'BestRound':>10}")
print("-" * 57)

for test_year in TEST_YEARS:
    train_years = list(range(TRAIN_START, test_year))
    all_train_dfs = [load_year(y) for y in train_years]
    val_df   = all_train_dfs[-1]                                  # Y-1 for early stopping
    train_df = pd.concat(all_train_dfs[:-1], ignore_index=True)   # 2015..Y-2

    dtrain_es = to_dmatrix(train_df, FEATURE_COLS)
    dval_es   = to_dmatrix(val_df,   FEATURE_COLS)

    model_es = xgb.train(
        XGB_PARAMS, dtrain_es,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dval_es, "val")],
        early_stopping_rounds=EARLY_STOPPING,
        verbose_eval=False,
    )
    best_round = model_es.best_iteration + 1

    full_train_df = pd.concat(all_train_dfs, ignore_index=True)
    dtrain_full   = to_dmatrix(full_train_df, FEATURE_COLS)
    model_final = xgb.train(
        XGB_PARAMS, dtrain_full,
        num_boost_round=best_round,
        verbose_eval=False,
    )

    test_df = load_year(test_year)
    dtest   = to_dmatrix(test_df, FEATURE_COLS)
    probs   = np.clip(model_final.predict(dtest), 1e-7, 1 - 1e-7)
    y_true  = test_df["home_win"].values.astype(float)

    ll    = log_loss(y_true, probs)
    acc   = (((probs >= 0.5).astype(int)) == y_true.astype(int)).mean()
    brier = brier_score_loss(y_true, probs)

    results.append(dict(year=test_year, n_test=len(test_df),
                        logloss=ll, accuracy=acc, brier=brier, best_round=best_round))
    print(f"{test_year:>6} {len(test_df):>7} {ll:>10.5f} {acc:>10.4f} {brier:>10.5f} {best_round:>10}")

df_res = pd.DataFrame(results)
print("-" * 57)
print(f"{'MEAN':>6} {'':>7} {df_res['logloss'].mean():>10.5f} "
      f"{df_res['accuracy'].mean():>10.4f} {df_res['brier'].mean():>10.5f}")
print(f"{'STD':>6} {'':>7} {df_res['logloss'].std():>10.5f} "
      f"{df_res['accuracy'].std():>10.4f} {df_res['brier'].std():>10.5f}")


  Year  n_test    LogLoss   Accuracy      Brier  BestRound
---------------------------------------------------------
  2020     147    0.64274     0.6122    0.22574         65
  2021     209    0.61491     0.6507    0.21507        332
  2022     239    0.65762     0.6360    0.23223        285
  2023     260    0.61880     0.6577    0.21435         91
  2024     262    0.59886     0.7061    0.20480        167
---------------------------------------------------------
  MEAN            0.62658     0.6525    0.21844
   STD            0.02340     0.0346    0.01070


In [3]:
# Feature importance for last fold (train 2015-2023, test 2024)
imp = model_final.get_score(importance_type="gain")
imp_df = (pd.DataFrame(list(imp.items()), columns=["feature", "gain"])
          .sort_values("gain", ascending=False)
          .reset_index(drop=True))

print("Top 20 features by gain (train 2015-2023, test 2024):")
print(imp_df.head(20).to_string(index=False))

for elo_col in ["home_elo_pre", "away_elo_pre"]:
    row = imp_df[imp_df["feature"] == elo_col]
    if not row.empty:
        rank = row.index[0] + 1
        print(f"\n  {elo_col}: rank {rank}/{len(imp_df)},  gain={row['gain'].values[0]:.1f}")
    else:
        print(f"\n  {elo_col}: NOT USED by model")


Top 20 features by gain (train 2015-2023, test 2024):
                           feature      gain
                      home_elo_pre 14.274438
                      away_elo_pre 13.253610
             home_net_rtg_ewma_pre  9.499374
             away_net_rtg_ewma_pre  9.310798
                     away_p2_q_pre  9.231005
                     home_p1_q_pre  8.884207
   away_p5_days_since_last_dnp_pre  8.615544
                     away_p1_q_pre  7.988779
away_p6_days_since_last_played_pre  7.658806
      home_p2_played_last_game_pre  7.577857
      away_p5_played_last_game_pre  6.949141
        away_games_last_4_days_pre  6.378774
      home_p5_played_last_game_pre  6.224892
   away_p5_injury_present_flag_pre  6.148737
                home_p2_m_ewma_pre  6.016240
                     home_p2_q_pre  5.865362
                   home_is_b2b_pre  5.819810
             home_travel_miles_pre  5.765348
away_p3_days_since_last_played_pre  5.683492
                     away_p3_q_pre  5.619396

